# 02 Target and data splitting

Use the [verified prediction design](../reports/prediction_design.md) to prepare the binary target and freeze the chronological split. Each transaction is identified by `(batch_id, leg, event_type)`. The scoring clock is `transactions.timestamp`, the recorded event time. The release does not specify a timezone.

This notebook reads only the transaction keys, timestamp and label. It also reads batch and fraud metadata for the overlap audit. It saves the split assignments and boundary rules together with checksums so later notebooks can reuse them.

In [ ]:
from hashlib import file_digest
from itertools import combinations
from pathlib import Path
from tempfile import TemporaryDirectory
import json

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import yaml
from IPython.display import display

current = Path.cwd()
root = next(
    path for path in (current, *current.parents)
    if (path / "AGENTS.md").is_file() and (path / "Phase1_Dataset" / "dataset").is_dir()
)
data_dir = root / "Phase1_Dataset" / "dataset"
processed_dir = root / "processed"
source_path = data_dir / "transactions.parquet"
target_path = processed_dir / "transaction_target.parquet"
config_path = root / "Phase1_Dataset" / "code" / "config.yaml"
batch_path = data_dir / "batches.parquet"
fraud_path = data_dir / "fraud_events.parquet"
assignments_path = processed_dir / "split_assignments.parquet"
boundaries_path = processed_dir / "split_boundaries.json"

key = ["batch_id", "leg", "event_type"]
clock = "timestamp"
target = "is_fraud_ground_truth"
split_names = ("train", "validation", "test")
excluded_name = "excluded_boundary_batch"
categories = (*split_names, excluded_name)
requested_fractions = (0.70, 0.15, 0.15)

source = pq.read_table(source_path, columns=[*key, clock, target])
prepared = pq.read_table(target_path, columns=[*key, target])
assert source[target].type == pa.bool_() and prepared[target].type == pa.uint8()
assert all(source[column].null_count == 0 for column in [*key, clock, target])
assert all(source[column].equals(prepared[column]) for column in key)
binary_target = pc.cast(source[target], pa.uint8())
assert binary_target.equals(prepared[target])
assert set(pc.unique(binary_target).to_pylist()) == {0, 1}

# Keep only the projected columns and use Arrow-backed strings for the identifiers.
transactions = source.select([*key, clock]).append_column(target, binary_target).to_pandas(types_mapper=pd.ArrowDtype)
del source, prepared, binary_target
assert not transactions.duplicated(key).any(), "Transaction keys must be unique"
assert not transactions[key].isna().any().any()
transactions[clock] = pd.to_datetime(transactions[clock], format="%Y-%m-%dT%H:%M:%S", errors="raise")
assert transactions[clock].notna().all() and transactions[clock].dt.tz is None
transactions = transactions.sort_values([clock, *key], ignore_index=True)
assert transactions[clock].is_monotonic_increasing
total_rows = len(transactions)
print(f"Verified and sorted {total_rows:,} transactions with complete 0/1 labels.")

## Choose chronological boundaries and keep batches intact

Use the timestamp group boundaries closest in row count to 70% and 85% of the file. The nearest boundary is chosen using timestamps and row counts only. If two boundaries are equally close, choose the earlier one. Each cutoff is the first timestamp of the next time window.

A batch with events on both sides of a cutoff cannot stay intact in those windows. Keep all its rows in the manifest under `excluded_boundary_batch`. Use the other rows for train, validation, and test. This keeps source rows accounted for and preserves strict chronology between the three evaluation sets. Equal timestamps in the retained sets belong to a single split. An excluded row can share a timestamp with a retained row.

In [ ]:
event_times = transactions[clock].to_numpy()
# Each candidate row position is the start of a different timestamp group.
group_starts = np.flatnonzero(event_times[1:] != event_times[:-1]) + 1
cut_positions = [
    int(group_starts[np.abs(group_starts - total_rows * fraction).argmin()])
    for fraction in (0.70, 0.85)
]
assert 0 < cut_positions[0] < cut_positions[1] < total_rows
assert all(event_times[position - 1] < event_times[position] for position in cut_positions)
validation_start, test_start = [pd.Timestamp(event_times[position]) for position in cut_positions]

batch_spans = transactions.groupby("batch_id", sort=False)[clock].agg(first="min", last="max")
crosses_validation = (batch_spans["first"] < validation_start) & (batch_spans["last"] >= validation_start)
crosses_test = (batch_spans["first"] < test_start) & (batch_spans["last"] >= test_start)
excluded_ids = batch_spans.index[crosses_validation | crosses_test]

# side="right" places a timestamp equal to a cutoff in the new time window.
split_codes = np.searchsorted(event_times[cut_positions], event_times, side="right")
split_codes[transactions["batch_id"].isin(excluded_ids).to_numpy(dtype=bool)] = len(split_names)
transactions["split"] = pd.Categorical.from_codes(split_codes, categories=categories, ordered=True)

summary = transactions.groupby("split", observed=False).agg(
    rows=(target, "size"),
    batches=("batch_id", "nunique"),
    first_timestamp=(clock, "min"),
    last_timestamp=(clock, "max"),
    labelled_rows=(target, "count"),
    fraud_rows=(target, "sum"),
)
summary["missing_targets"] = summary["rows"] - summary["labelled_rows"]
summary["genuine_rows"] = summary["labelled_rows"] - summary["fraud_rows"]
retained_rows = int(summary.loc[list(split_names), "rows"].sum())
summary["share_of_source_pct"] = 100 * summary["rows"] / total_rows
summary["share_of_retained_pct"] = 100 * summary["rows"] / retained_rows
summary.loc[excluded_name, "share_of_retained_pct"] = np.nan

assert int(summary["rows"].sum()) == total_rows
assert summary["missing_targets"].eq(0).all()
assert summary.loc[list(split_names), ["genuine_rows", "fraud_rows"]].gt(0).all().all()
assert np.allclose(summary.loc[list(split_names), "rows"] / retained_rows, requested_fractions, atol=0.02, rtol=0)
assert summary.loc["train", "last_timestamp"] < validation_start <= summary.loc["validation", "first_timestamp"]
assert summary.loc["validation", "last_timestamp"] < test_start <= summary.loc["test", "first_timestamp"]
assert list(transactions["split"].cat.categories) == list(categories)
assert transactions["split"].notna().all()
assert set(transactions.loc[transactions["split"].eq(excluded_name), "batch_id"]) == set(excluded_ids)
assert transactions.groupby("batch_id", sort=False)["split"].nunique().eq(1).all()

# Strict interval checks above imply zero retained timestamp ties across splits.
print(f"Validation window starts {validation_start.isoformat()}. Test window starts {test_start.isoformat()}.")
print(f"Boundary-spanning batches: {crosses_validation.sum():,} at validation, "
      f"{crosses_test.sum():,} at test, {len(excluded_ids):,} distinct.")
display(summary.round(3))

## Check label availability at the cutoffs

TRACE-X gives us complete retrospective labels. The simulator copies a batch's ground truth to all events and separately samples a detection flag. The verified design identifies no confirmation timestamp or delay. We can count labelled rows in the earlier windows, but cannot verify which labels would already have been available for a real training run at either cutoff. A missing timestamp for a label's availability tells us nothing about whether the label was delayed.

Use the complete labels only for this retrospective split. If actual label-availability times become available, apply `label_available_at <= training_cutoff` separately for each fit and exclude pending or unknown labels from that fit. The test cutoff below describes what would be available for a possible train-plus-validation fit. There is no fit in this notebook. [Prediction design](../reports/prediction_design.md#target-and-label-availability) · [Generator](../Phase1_Dataset/code/generator.py#L299)

In [ ]:
cutoff_rows = []
for cutoff, eligible_splits in [
    (validation_start, ["train"]),
    (test_start, ["train", "validation"]),
]:
    eligible = summary.loc[eligible_splits]
    candidate_rows = int(eligible["rows"].sum())
    offline_labels = int(eligible["labelled_rows"].sum())
    assert candidate_rows == offline_labels
    cutoff_rows.append({
        "cutoff": cutoff.isoformat(),
        "earlier_splits": eligible_splits,
        "candidate_rows": candidate_rows,
        "offline_labels": offline_labels,
        "verified_available_by_cutoff": None,
        "availability": "Unknown. The release has no label-availability timestamp.",
    })
display(pd.DataFrame(cutoff_rows))

## Check for duplicates, shared fraud records, and overlap

The unique composite key rules out exact full-row duplicates because two identical rows would have the same key. It also rules out a transaction appearing in two splits. Fraud records are keyed by batch, so audit their overlap at the batch level. For campaign overlap, use the configured production-date windows. A batch in a window was exposed to the configured timing rule in the supplied simulator. The release has no per-batch campaign ID or ring ID. `in_ring` supports a check for shared ring-flagged manufacturers. [Schema summary](../reports/schema_summary.md#keys-and-relationships) · [Configuration](../Phase1_Dataset/code/config.yaml#L93) · [Fraud assignment](../Phase1_Dataset/code/generator.py#L167)

In [ ]:
assert transactions.duplicated(key).sum() == 0
# Zero repeated complete keys means zero exact duplicates and zero cross-split transaction ID overlap.
batches = pd.read_parquet(batch_path, columns=["batch_id", "manufacturer_id", "production_date"])
fraud = pd.read_parquet(fraud_path, columns=["batch_id", target, "in_ring"])
assert not batches[["batch_id", "manufacturer_id", "production_date"]].isna().any().any()
assert not fraud.isna().any().any()
batch_labels = transactions.groupby("batch_id", sort=False)[target]
assert batch_labels.nunique().eq(1).all()
batch_assignment = transactions.groupby("batch_id", sort=False).agg(
    split=("split", "first"), transaction_target=(target, "first")
).reset_index()
batch_audit = batch_assignment.merge(batches, on="batch_id", how="inner", validate="one_to_one")
batch_audit = batch_audit.merge(fraud, on="batch_id", how="inner", validate="one_to_one")
assert len(batch_audit) == len(batch_assignment) == len(batches) == len(fraud)
assert batch_audit["transaction_target"].eq(batch_audit[target].astype("uint8")).all()
batch_audit["production_date"] = pd.to_datetime(
    batch_audit["production_date"], format="%Y-%m-%d", errors="raise"
)
retained = batch_audit.loc[batch_audit["split"].isin(split_names)]
windows = yaml.safe_load(config_path.read_text(encoding="utf-8"))["temporal_campaigns"]

campaign_rows = []
for campaign in windows:
    start, end = map(pd.Timestamp, campaign["window"])
    covered = retained["production_date"].between(start, end, inclusive="both")
    campaign_rows.append({
        "production_window": f"{start.date()} through {end.date()}",
        **{name: int((covered & retained["split"].eq(name)).sum()) for name in split_names},
    })
display(pd.DataFrame(campaign_rows))

audit_groups = {name: retained.loc[retained["split"].eq(name)] for name in split_names}
overlap_rows = []
for left, right in combinations(split_names, 2):
    first, second = audit_groups[left], audit_groups[right]
    shared_batches = set(first["batch_id"]) & set(second["batch_id"])
    shared_fraud_batches = set(first.loc[first[target], "batch_id"]) & set(second.loc[second[target], "batch_id"])
    shared_manufacturers = set(first["manufacturer_id"]) & set(second["manufacturer_id"])
    shared_ring_manufacturers = (
        set(first.loc[first["in_ring"], "manufacturer_id"])
        & set(second.loc[second["in_ring"], "manufacturer_id"])
    )
    shared_campaigns = [row["production_window"] for row in campaign_rows if row[left] and row[right]]
    assert not shared_batches and not shared_fraud_batches
    overlap_rows.append({
        "pair": f"{left} / {right}",
        "shared_transaction_ids": 0,
        "shared_batch_fraud_records": len(shared_batches),
        "shared_positive_fraud_batches": len(shared_fraud_batches),
        "shared_manufacturers": len(shared_manufacturers),
        "shared_ring_flagged_manufacturers": len(shared_ring_manufacturers),
        "shared_campaign_windows": shared_campaigns,
    })
display(pd.DataFrame(overlap_rows))
print("Exact duplicate transaction rows: 0.")

### Is another group-based evaluation required?

The main evaluation already keeps batches intact. It answers how well a future model would perform on later events under this release's mix of manufacturers and campaigns. Use a separate chronological, manufacturer-disjoint evaluation if claiming performance on unseen manufacturers, and hold out entire campaign windows if claiming transfer to unseen campaigns. Shared campaign windows and ring-flagged manufacturers can make the primary results optimistic for those claims. The release lacks incident and ring IDs, so any proxy-based challenge should say what it can verify. Keep this primary split fixed for comparisons. The supplementary evaluation is outside this notebook's scope.

## Save the fixed split

Use the saved assignment table by transaction key in later notebooks. Keep the target in its existing separate file. Use `train`, `validation`, and `test` as the evaluation sets and exclude `excluded_boundary_batch` from fitting and scoring. The boundary JSON records the exact cutoff rule, reported counts, audit results, label timing limitation, and input fingerprints. Use the assignment fingerprint in the JSON to confirm that the two saved files belong to the same run.

In [ ]:
def sha256(path):
    with path.open("rb") as source_file:
        return file_digest(source_file, "sha256").hexdigest()


# Store a portable timestamp type and plain split names in the manifest.
assignments = pa.table({
    **{column: pa.array(transactions[column]) for column in key},
    clock: pa.array(transactions[clock], type=pa.timestamp("us")),
    "split": pa.array(transactions["split"].astype("string"), type=pa.string()),
})
with TemporaryDirectory(dir=processed_dir) as staging_dir:
    staged_assignments = Path(staging_dir) / assignments_path.name
    staged_boundaries = Path(staging_dir) / boundaries_path.name
    pq.write_table(assignments, staged_assignments, compression="zstd")
    assert pq.read_table(staged_assignments).equals(assignments)

    metadata = {
        "assignment_file": assignments_path.name,
        "assignment_sha256": sha256(staged_assignments),
        "input_sha256": {
            path.relative_to(root).as_posix(): sha256(path)
            for path in (source_path, target_path, batch_path, fraud_path, config_path)
        },
        "transaction_key": key,
        "target_column": target,
        "prediction_timestamp": clock,
        "timestamp_timezone": None,
        "requested_fractions": dict(zip(split_names, requested_fractions, strict=True)),
        "selection": "Nearest cumulative row count to 70% and 85% between complete timestamp groups. Break ties toward the earlier boundary.",
        "validation_start_inclusive": validation_start.isoformat(),
        "test_start_inclusive": test_start.isoformat(),
        "interval_rule": "Train is before validation start. Validation starts inclusively and ends before test start. Test starts inclusively.",
        "excluded_assignment": excluded_name,
        "exclusion_rule": "Assign every row of a batch crossing either cutoff to excluded_boundary_batch. Keep excluded rows in the manifest.",
        "source_rows": total_rows,
        "retained_rows": retained_rows,
        "excluded_batches_at_validation_cutoff": int(crosses_validation.sum()),
        "excluded_batches_at_test_cutoff": int(crosses_test.sum()),
        "label_policy": "Retrospective labels only. Availability at calendar cutoffs is unknown.",
        "label_cutoff_checks": cutoff_rows,
        "split_summary": json.loads(summary.to_json(orient="index", date_format="iso")),
        "campaign_batch_counts": campaign_rows,
        "overlap_checks": overlap_rows,
        "exact_duplicate_rows": 0,
    }
    staged_boundaries.write_text(json.dumps(metadata, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    assert json.loads(staged_boundaries.read_text(encoding="utf-8")) == metadata
    staged_assignments.replace(assignments_path)
    staged_boundaries.replace(boundaries_path)

assert sha256(assignments_path) == json.loads(boundaries_path.read_text(encoding="utf-8"))["assignment_sha256"]
print(f"Saved {assignments_path.relative_to(root)} and {boundaries_path.relative_to(root)}.")
print("All split assertions passed.")